# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading, exploring, and analyzing the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

Below, we will list all record sets, the fields they contain, and their associated Croissant `@id`s. This helps you understand the data structure and what you can access.

In [ ]:
# List available record sets and their fields, referencing by '@id' as required.

record_set_ids = []
record_set_id_to_fields = {}

for record_set in metadata.record_sets:
    print(f"Record Set: {record_set.name}")
    print(f"  @id: {record_set.id}")
    record_set_ids.append(record_set.id)
    field_ids = []
    for field in record_set.fields:
        print(f"    Field: {field.name} (@id: {field.id}, type: {field.data_type})")
        field_ids.append(field.id)
    record_set_id_to_fields[record_set.id] = field_ids
    print("")

if not record_set_ids:
    print("No record sets found in metadata.")
else:
    print(f"Record set ids found: {record_set_ids}")

## 3. Data Extraction
Load data from all available record sets into pandas DataFrames for analysis.

Below, we load each record set using its `@id` and collect the records in a dictionary of DataFrames. We display the columns and show the first few records for each.

In [ ]:
# Extract data from each record set
dataframes = {}

if not record_set_ids:
    print("No record sets to extract data from.")
else:
    for record_set_id in record_set_ids:
        print(f"Loading record set: {record_set_id}")
        records = list(dataset.records(record_set=record_set_id))
        if not records:
            print(f"  No records found for record set: {record_set_id}")
            continue
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"  Columns: {df.columns.tolist()}")
        display(df.head(3))

    # Pick the first record set for next steps if available
    if record_set_ids:
        main_record_set_id = record_set_ids[0]
    else:
        main_record_set_id = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. 

Below, we select a numeric field and perform filtering, normalization, and grouping. All references use the field's `@id`.

In [ ]:
import numpy as np

# We'll demonstrate EDA on the first record set, if available and if data is present
if not dataframes:
    print("No dataframes available for EDA.")
elif not main_record_set_id:
    print("No main record set ID selected.")
else:
    df = dataframes[main_record_set_id]
    # Find a numeric field by looking up its data_type
    numeric_field_id = None
    group_field_id = None

    for record_set in metadata.record_sets:
        if record_set.id == main_record_set_id:
            for field in record_set.fields:
                # Typical Croissant data types: 'schema:Float', 'schema:Integer', etc.
                if field.data_type in ["schema:Float", "schema:Integer", "Float", "Integer"] and field.id in df.columns:
                    numeric_field_id = field.id
                    break
            # For grouping, pick the first non-numeric field
            for field in record_set.fields:
                if field.data_type not in ["schema:Float", "schema:Integer", "Float", "Integer"] and field.id in df.columns:
                    group_field_id = field.id
                    break

    if numeric_field_id is None:
        print("No numeric field found for EDA.")
    else:
        print(f"Using numeric field: {numeric_field_id}")
        
        # Ensure numeric dtype
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        threshold = np.nanpercentile(df[numeric_field_id], 75) if df[numeric_field_id].notnull().sum() > 0 else 10
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head(3))
        
        # Normalization
        if filtered_df.shape[0] > 0:
            field_norm = f"{numeric_field_id}_normalized"
            filtered_df[field_norm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
            print(f"Normalized {numeric_field_id} for filtered records:")
            display(filtered_df[[numeric_field_id, field_norm]].head(3))
            # Grouping by another field (categorical)
            if group_field_id is not None and group_field_id in filtered_df.columns:
                grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].agg(['mean', 'count']).reset_index()
                print(f"Grouped data by {group_field_id} (showing mean and count):")
                display(grouped_df.head(3))
            else:
                print("No suitable group field found for grouping.")
        else:
            print("No records passed the filtering threshold.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. 

We'll show some basic plots of the numeric field, including distribution and relationship to group/categorical field if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualization if data exists
if 'filtered_df' in locals() and not filtered_df.empty and numeric_field_id is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(filtered_df[numeric_field_id].dropna(), kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field_id} (filtered)")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field_id is not None and group_field_id in filtered_df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Not enough data for visualization.")

## 6. Conclusion
In this notebook, we loaded the FAIR² dataset via its Croissant schema using `mlcroissant`, explored its record sets and fields via their `@id`s, and demonstrated basic data processing and analysis steps. For robust analyses, refer to field definitions and units in the Croissant metadata, and consult domain experts as needed. All data operations were referenced by `@id`, providing clarity and reproducibility.